# Dar_030 — Local time, seasonal phase, terrain and dated geometry

**Primary target:** outline Stage 3 — environmental conditions + mission/activity + crew + equipment + applicable requirements → adequacy/margins/failed and uncertain requirements → preferences only after mandatory checks pass.

required Stage 3 work precedes the unchanged research P1/P2/P3 tags. This notebook does not invent mission inputs or turn reference data into local measurements.

## Questions
1. How many local-hour × Ls bins actually have thermal data in each cohort?
2. How different are observed seasonal medians at a matched local hour, and are samples comparable?
3. Does a cyclic local-time/Ls model transfer to a held-out seasonal group better than its training-mean baseline?
4. Do terrain–temperature relationships retain their signs after solar/local-time controls, across seasonal groups?
5. How do Sun/Earth reference elevations and local solar hour vary through monthly samples in 2020?
6. What below-horizon sample spans and censoring occur in the existing November reference history, without claiming physical darkness/contact durations?


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode=True
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'outline.md').is_file() and (p/'notebooks').is_dir())
sys.path[:0]=[str(ROOT / "env_algo_task/EDA/shared"), str(ROOT / "env_algo_task/EDA/Moon/Stage3_Requirements"),str(ROOT)]
import pandas as pd
from IPython.display import display
import golden_stage3_eda as methods


## Declared design and sources

The implementation pins accepted completed runs and verifies consumed output hashes. Missing records remain missing. Units, local-time/season support, spatial/temporal applicability and reference/observed/modelled roles are not interchangeable.


In [2]:
OUT=methods.execute('time_terrain','Moon/Stage3_Requirements/Dar_030_stage3_time_season_terrain.ipynb')
findings=json.loads((OUT/'findings.json').read_text(encoding='utf-8'))
display(findings)


{
  "local_hour_ls_rows": 3024,
  "supported_local_hour_ls_rows": 1032,
  "matched_hour_contrasts_with_two_ls": 191,
  "cyclic_supported_folds": 108,
  "cyclic_folds_better_than_training_mean": 83,
  "terrain_phase_comparisons": 2142,
  "annual_reference_geometry_rows": 4704,
  "below_reference_horizon_sample_spans": 25,
  "holds": [
    "Ls and local solar hour are cyclic phase coordinates, not calendar-month temperature observations",
    "Matched local-hour summaries do not guarantee identical sample centres across Ls",
    "Aggregate-bin CV is a diagnostic, not independent site prediction or an equipment response model",
    "Annual geometry samples only first seven days/month; terrain-free point-centre geometry, not terrain illumination/contact",
    "November intervals are first-to-last sample spans with censoring, not measured continuous darkness"
  ]
}
Completed: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\env_algo_task\data\processed\analysis\stage3_req

{'local_hour_ls_rows': 3024,
 'supported_local_hour_ls_rows': 1032,
 'matched_hour_contrasts_with_two_ls': 191,
 'cyclic_supported_folds': 108,
 'cyclic_folds_better_than_training_mean': 83,
 'terrain_phase_comparisons': 2142,
 'annual_reference_geometry_rows': 4704,
 'below_reference_horizon_sample_spans': 25,
 'holds': ['Ls and local solar hour are cyclic phase coordinates, not calendar-month temperature observations',
  'Matched local-hour summaries do not guarantee identical sample centres across Ls',
  'Aggregate-bin CV is a diagnostic, not independent site prediction or an equipment response model',
  'Annual geometry samples only first seven days/month; terrain-free point-centre geometry, not terrain illumination/contact',
  'November intervals are first-to-last sample spans with censoring, not measured continuous darkness']}

### local_hour_season_coverage.csv

Phase-bin averages are not mission-time minima/maxima. Empty bins retain NaN.


In [3]:
table=pd.read_csv(OUT/'local_hour_season_coverage.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 126


,region,cohort,seasonal_ls_bin,expected_hour_bins,observed_hour_bins,min_bin_median_K,max_bin_median_K,span_of_observed_hour_bin_medians_K
0,A3CLR22_10_Leibnitz_Beta_Plateau,available,30,24,16,62.087219,159.145722,97.058503
1,A3CLR22_10_Leibnitz_Beta_Plateau,available,90,24,17,56.202648,230.586350,174.383702
2,A3CLR22_10_Leibnitz_Beta_Plateau,available,150,24,17,60.470472,192.162872,131.692399
3,A3CLR22_10_Leibnitz_Beta_Plateau,available,210,24,18,69.284531,220.530578,151.246047
4,A3CLR22_10_Leibnitz_Beta_Plateau,available,270,24,18,77.013832,183.267838,106.254006
5,A3CLR22_10_Leibnitz_Beta_Plateau,available,330,24,17,72.224186,174.330650,102.106465
6,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,30,24,3,105.390278,159.145722,53.755444
7,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,90,24,1,58.984043,58.984043,0.000000
8,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,150,24,3,66.364182,76.470116,10.105935
9,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,210,24,4,78.148968,84.727314,6.578346


### matched_hour_seasonal_contrasts.csv

Matched hour does not guarantee matched physical centres; count and cohort differences remain visible.


In [4]:
table=pd.read_csv(OUT/'matched_hour_seasonal_contrasts.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 504


,region,cohort,local_time_bin_h,observed_ls_groups,minimum_bin_median_K,maximum_bin_median_K,minimum_bin_centres,maximum_bin_centres,matched_hour_ls_span_K
0,A3CLR22_10_Leibnitz_Beta_Plateau,available,0,4,61.002159,85.179314,0,4422,24.177155
1,A3CLR22_10_Leibnitz_Beta_Plateau,available,1,4,59.723383,84.886676,0,4422,25.163293
2,A3CLR22_10_Leibnitz_Beta_Plateau,available,2,4,60.103182,81.698558,0,4422,21.595376
3,A3CLR22_10_Leibnitz_Beta_Plateau,available,3,4,62.413775,77.285697,0,4422,14.871923
4,A3CLR22_10_Leibnitz_Beta_Plateau,available,4,4,60.845356,72.685673,0,4422,11.840317
5,A3CLR22_10_Leibnitz_Beta_Plateau,available,5,4,59.651956,70.339138,0,4422,10.687181
6,A3CLR22_10_Leibnitz_Beta_Plateau,available,6,5,58.620146,73.202495,0,4422,14.582349
7,A3CLR22_10_Leibnitz_Beta_Plateau,available,7,4,59.046107,72.224186,0,4422,13.178080
8,A3CLR22_10_Leibnitz_Beta_Plateau,available,8,4,58.984043,90.305496,0,4422,31.321453
9,A3CLR22_10_Leibnitz_Beta_Plateau,available,9,5,65.050277,98.965027,0,4422,33.914750


### held_out_ls_cyclic_diagnostics.csv

Fixed five-column cyclic basis, whole-Ls holdout, training mean baseline. Not a validated temperature forecaster or CNN training dataset.


In [5]:
table=pd.read_csv(OUT/'held_out_ls_cyclic_diagnostics.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 126


,region,cohort,held_out_ls,train_bins,test_bins,cyclic_mae_K,training_mean_baseline_mae_K,status
0,A3CLR22_10_Leibnitz_Beta_Plateau,available,30,87,16,21.002232,25.208166,aggregate_diagnostic
1,A3CLR22_10_Leibnitz_Beta_Plateau,available,90,86,17,19.915128,38.568660,aggregate_diagnostic
2,A3CLR22_10_Leibnitz_Beta_Plateau,available,150,86,17,19.881278,33.999435,aggregate_diagnostic
3,A3CLR22_10_Leibnitz_Beta_Plateau,available,210,85,18,18.246311,22.747596,aggregate_diagnostic
4,A3CLR22_10_Leibnitz_Beta_Plateau,available,270,85,18,19.941253,31.886197,aggregate_diagnostic
5,A3CLR22_10_Leibnitz_Beta_Plateau,available,330,86,17,17.168070,23.180875,aggregate_diagnostic
6,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,30,14,3,45.307943,44.156754,aggregate_diagnostic
7,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,90,16,1,42.503349,35.006340,aggregate_diagnostic
8,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,150,14,3,10.765526,26.188985,aggregate_diagnostic
9,A3CLR22_10_Leibnitz_Beta_Plateau,common_available,210,13,4,16.088556,14.067339,aggregate_diagnostic


### terrain_temperature_phase_sign_support.csv

Phase/cohort/control-qualified sign stability, not a universal slope/roughness correction.


In [6]:
table=pd.read_csv(OUT/'terrain_temperature_phase_sign_support.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 108


,cohort,predictor,control_model,seasonal_ls_bin,comparisons,supported,median_conditional_rho,minimum_rho,maximum_rho,negative_fraction_supported
0,available,sdem_elevation_m,solar_rank,30,21,20,0.329218,-0.481418,0.754629,0.300000
1,available,sdem_elevation_m,solar_rank,90,7,7,0.013270,-0.255843,0.489821,0.428571
2,available,sdem_elevation_m,solar_rank,150,21,20,-0.090103,-0.423008,0.345575,0.600000
3,available,sdem_elevation_m,solar_rank,210,28,28,0.021257,-0.600850,0.665412,0.464286
4,available,sdem_elevation_m,solar_rank,270,21,19,-0.269383,-0.546749,0.397907,0.684211
5,available,sdem_elevation_m,solar_rank,330,21,21,-0.096684,-0.604465,0.446907,0.619048
6,available,sdem_elevation_m,solar_rank_local_time_basis,30,21,20,0.356632,-0.481434,0.805479,0.350000
7,available,sdem_elevation_m,solar_rank_local_time_basis,90,7,7,-0.119252,-0.247384,0.511629,0.571429
8,available,sdem_elevation_m,solar_rank_local_time_basis,150,21,20,-0.006590,-0.574091,0.275326,0.550000
9,available,sdem_elevation_m,solar_rank_local_time_basis,210,28,28,0.116777,-0.628575,0.609724,0.464286


### monthly_reference_geometry_summary.csv

First seven days of each 2020 month at six-hour cadence; reference sphere, no terrain horizon or relay/link model.


In [7]:
table=pd.read_csv(OUT/'monthly_reference_geometry_summary.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 168


,region,target,month,samples,minimum_elevation_deg,maximum_elevation_deg,above_sphere_horizon_fraction
0,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,1,28,-3.288387,-0.049690,0.000000
1,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,2,28,-2.804772,5.595873,0.535714
2,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,3,28,-1.359247,7.928599,0.785714
3,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,4,28,3.655111,10.080489,1.000000
4,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,5,28,7.768168,10.129235,1.000000
5,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,6,28,2.213416,10.232544,1.000000
6,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,7,28,-1.292187,8.395611,0.821429
7,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,8,28,-2.968705,2.789702,0.285714
8,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,9,28,-2.837479,-0.629599,0.000000
9,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH,10,28,-2.788137,2.947041,0.357143


### november_reference_below_horizon_sample_spans.csv

First-to-last sample spans and boundary censoring; not continuous darkness or actual communications outages.


In [8]:
table=pd.read_csv(OUT/'november_reference_below_horizon_sample_spans.csv')
print('Rows:',len(table))
display(table.head(30))


Rows: 25


,first_sample_utc,last_sample_utc,sample_count,first_to_last_sample_span_hours,left_censored,right_censored,region,target
0,2020-11-01 00:00:00+00:00,2020-11-01 11:00:00+00:00,12,11.0,True,False,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH
1,2020-11-19 12:00:00+00:00,2020-11-28 16:00:00+00:00,221,220.0,False,False,A3CLR22_10_Leibnitz_Beta_Plateau,EARTH
2,2020-11-04 09:00:00+00:00,2020-11-20 17:00:00+00:00,393,392.0,False,False,A3CLR22_10_Leibnitz_Beta_Plateau,SUN
3,2020-11-01 00:00:00+00:00,2020-11-01 07:00:00+00:00,8,7.0,True,False,A3CLR22_11_Malapert_Massif,EARTH
4,2020-11-19 09:00:00+00:00,2020-11-28 13:00:00+00:00,221,220.0,False,False,A3CLR22_11_Malapert_Massif,EARTH
5,2020-11-06 22:00:00+00:00,2020-11-23 09:00:00+00:00,396,395.0,False,False,A3CLR22_11_Malapert_Massif,SUN
6,2020-11-01 00:00:00+00:00,2020-11-05 08:00:00+00:00,105,104.0,True,False,A3CLR22_12_deGerlache_Kocher_Massif,EARTH
7,2020-11-15 17:00:00+00:00,2020-11-30 23:00:00+00:00,367,366.0,False,True,A3CLR22_12_deGerlache_Kocher_Massif,EARTH
8,2020-11-01 00:00:00+00:00,2020-11-03 21:00:00+00:00,70,69.0,True,False,A3CLR22_12_deGerlache_Kocher_Massif,SUN
9,2020-11-16 19:00:00+00:00,2020-11-30 23:00:00+00:00,341,340.0,False,True,A3CLR22_12_deGerlache_Kocher_Massif,SUN


## Findings and algorithm boundary

Findings above are executed outputs, not an adopted safety/scoring policy. The consolidated measured results and accepted/reference-only/deferred/rejected contribution decisions live in `EDA_Notes.md` and `purposal.md`. required Stage 3 input gaps remain the next work, not a reason to promote unrelated extra datasets. Unsupported operational checks remain Unknown.
